# Preliminary

In [ ]:
%reload_ext autoreload
%autoreload 2
import qiskit_metal
print(qiskit_metal.about())

In [ ]:
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import LaunchpadWirebondDriven




In [ ]:
s = 10

ratio = 0.2

w = 6#((1-ratio)/2)*s


In [ ]:
import numpy as np
from collections import OrderedDict
from IPython import display
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, Headings

resonator_design = designs.DesignPlanar()


# if you disable the next line, then you will need to delete a component [<component>.delete()] before recreating it
resonator_design.overwrite_enabled = True
resonator_design._chips['main']['size']['size_z'] = '-750um'
resonator_design.variables['cpw_width'] = f'{s} um'
resonator_design.variables['cpw_gap'] = f'{round(w, 3)} um'
print(f"CPW parameters:\nS={resonator_design.variables['cpw_width']}\nW={resonator_design.variables['cpw_gap']}\nRatio(S/S+2W)={ratio}\nH={resonator_design.get_chip_size()['size_z']}")
resonator_design._chips['main']['size']['size_x'] = '9mm'
resonator_design._chips['main']['size']['size_y'] = '9mm'

##############
length = '3000um'
##############
pad_width = '80um'
pad_height = '80um'
P1_Q = LaunchpadWirebondDriven(resonator_design, 'P1_Q', options = dict(pos_x='0um', pos_y='0um', orientation='0', lead_length='40um', pad_width=pad_width, pad_height=pad_height))

P2_Q = LaunchpadWirebondDriven(resonator_design, 'P2_Q', options = dict(pos_x=length, pos_y='0um', orientation='180', lead_length='40um', pad_width=pad_width, pad_height=pad_height))

cpw_feedline = RouteStraight(resonator_design, 'cpw_feedline', options= Dict(pin_inputs=Dict(start_pin=Dict(component='P1_Q', pin='tie'), end_pin=Dict(component='P2_Q', pin='tie')) ,fillet="99um", hfss_wire_bonds=True))

In [ ]:
qiskit_metal.view(resonator_design)

In [ ]:
from qiskit_metal.analyses.simulation import ScatteringImpedanceSim

scatimsim = ScatteringImpedanceSim(resonator_design, renderer_name='hfss')


In [ ]:
scatimsim.setup

In [ ]:
scatimsim.setup.freq_ghz = 9
scatimsim.setup.basis_order = -1
scatimsim.max_passes = 20 
scatimsim.min_passes = 1
scatimsim.max_converged = 2
scatimsim.max_delta_s = 0.005



In [ ]:
scatimsim.setup_update(max_delta_s=0.001,sweep_setup={'name': 'Sweep',
  'start_ghz': 2.0,
  'stop_ghz': 10.0,
  'count': 10001,
  'step_ghz': None,
  'type': 'Interpolating',
  'save_fields': True, 
  })
scatimsim.setup

In [ ]:
resonator_design.components.P1_Q

In [ ]:
scatimsim.renderer.options["x_buffer_width_mm"] = 0.1
scatimsim.renderer.options["y_buffer_width_mm"] = 0.1

In [ ]:
resonator_design.rebuild()
scatimsim._render(
            name= "Test_with_launch_ports_wo_launch_3_50_1_Mixed",
            components=[],
            open_terminations=[], 
            port_list=[('P1_Q', 'in', 50), ('P2_Q', 'in', 50)], 
            jj_to_port=[], 
            ignored_jjs=[],
            box_plus_buffer = True)

In [ ]:
%matplotlib inline
s_values, fig = scatimsim.get_scattering()


In [ ]:
s_values

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
ax.plot(list(s_values.index), np.abs(s_values['S11']) + np.abs(s_values['S21']) )

In [ ]:
import matplotlib.pyplot as plt
def plot_multiple_complex_arrays(*complex_arrays, labels=None):
    """
    Plots multiple arrays of complex numbers on a single 2D plane.
    
    Parameters:
    *complex_arrays : Arbitrary number of lists containing complex numbers.
    labels (list)   : Optional list of strings to label each array in the legend.
    """
    plt.figure(figsize=(9, 9))
    
    # Iterate through each array passed to the function
    for i, complex_numbers in enumerate(complex_arrays):
        real_parts = [z.real for z in complex_numbers]
        imag_parts = [z.imag for z in complex_numbers]
        
        # Determine the label for the legend
        if labels and i < len(labels):
            current_label = labels[i]
        else:
            current_label = f'Array {i + 1}'
            
        # Plot this specific array (Matplotlib automatically assigns a new color)
        plt.scatter(real_parts, imag_parts, marker='o', label=current_label, zorder=3, s=1)

        # Annotate each point with its mathematical value
        # for z in complex_numbers:
        #     point_label = f"{z.real:g}{'+' if z.imag >= 0 else ''}{z.imag:g}j"
        #     plt.annotate(point_label, (z.real, z.imag), textcoords="offset points", 
        #                  xytext=(5, 5), ha='left', fontsize=8)

    # Configure axes and grid
    plt.axhline(0, color='black', linewidth=1.5, zorder=1) # X-axis
    plt.axvline(0, color='black', linewidth=1.5, zorder=1) # Y-axis
    plt.grid(True, linestyle='--', alpha=0.6, zorder=0)

    # Set labels, title, and legend
    plt.xlabel('Real Axis')
    plt.ylabel('Imaginary Axis')
    plt.title('Complex Plane (Multiple Arrays)')
    plt.legend()

    # Ensure the aspect ratio is 1:1
    plt.axis('equal')

    # Display the plot
    plt.show()

plot_multiple_complex_arrays(s_values['S21'], s_values['S11'], labels=['S21', 'S11'])

In [ ]:
scatimsim.renderer.get_convergences()

In [ ]:
z_values, fig_z = scatimsim.get_impedance()

# Automation

In [1]:
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import LaunchpadWirebondDriven
from qiskit_metal.qlibrary.terminations.launchpad_wb_coupled import LaunchpadWirebondCoupled


import numpy as np
from collections import OrderedDict
from IPython import display
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, Headings

def create_transmission_line_design():
    resonator_design = designs.DesignPlanar()

    # if you disable the next line, then you will need to delete a component [<component>.delete()] before recreating it
    resonator_design.overwrite_enabled = True

    resonator_design.variables['cpw_width'] = '10 um'
    resonator_design.variables['cpw_gap'] = '6 um'
    resonator_design._chips['main']['size']['size_x'] = '9mm'
    resonator_design._chips['main']['size']['size_y'] = '6.5mm'


    P1_Q = LaunchpadWirebondDriven(resonator_design, 'P1_Q', options = dict(pos_x='-1500um', pos_y='0um', orientation='0', lead_length='30um'))

    P2_Q = LaunchpadWirebondDriven(resonator_design, 'P2_Q', options = dict(pos_x='1500um', pos_y='0um', orientation='180', lead_length='30um'))

    cpw_feedline = RouteStraight(resonator_design, 'cpw_feedline', options= Dict(pin_inputs=Dict(start_pin=Dict(component='P1_Q', pin='tie'), end_pin=Dict(component='P2_Q', pin='tie')) ,fillet='0um'))

    resonator_design.rebuild()

    return resonator_design


12:08PM 01s WARNING [_maybe_warn_lite_flip]: [FutureWarning] quantum-metal v0.7.0 will move PySide6, qdarkstyle, pyaedt, pyEPR-quantum, and gmsh out of base dependencies into opt-in extras. To preserve the current v0.6.x install behaviour, run `pip install 'quantum-metal[full]'` before upgrading. See ROADMAP.md and docs/migration-to-v0.7.0.rst for details. Set QISKIT_METAL_SUPPRESS_LITE_FLIP_WARNING=1 to silence.


In [ ]:
from qiskit_metal.analyses import ScatteringImpedanceSim
import matplotlib.pyplot as plt
def simulate_s_params(design, design_name, selection, open_pins, port_list, box_plus_buffer):
    em = ScatteringImpedanceSim(design, "hfss")


    em.setup.name = "Sweep_DrivenModal_setup"
    em.setup.freq_ghz = 6.0  # Try to keep this at the center of the swept frequency range for 'fast' sweeps and at the largest frequency for interpolating sweep for the best results
    em.setup.max_delta_s = 0.005  # This is necessary to get good results if interpolating sweep is not working for you
    em.setup.max_passes = 20
    em.setup.min_passes = 2
    em.setup.basis_order = -1  # Mixed order

    print(em.setup)
    em.setup.sweep_setup.name="Sweep_options_dm_sweep"
    em.setup.sweep_setup.start_ghz=2.0
    em.setup.sweep_setup.stop_ghz=10.0
    em.setup.sweep_setup.count=10001
    em.setup.sweep_setup.type="Interpolating"
    print(em.setup.sweep_setup)
    em.renderer.start()
    em.setup.renderer.options['x_buffer_width_mm'] = 0.1
    em.setup.renderer.options['y_buffer_width_mm'] = 0.1
    em._render(name = design_name,
                selection=selection,
                solution_type="drivenmodal",
                vars_to_initialize=em.setup.vars,
                open_pins=open_pins,
                port_list=port_list,
                box_plus_buffer=box_plus_buffer)



    em._analyze()

    freqs, Pcurves, Pparams = em.renderer.get_params([f"S{i+1}1" for i in range(len(port_list))])

    conv_t, conv_f, text = em.renderer.get_convergences()

    return Pparams, conv_t, conv_f, text
    
    
    

In [3]:
resonator_design = create_transmission_line_design()

In [5]:
%matplotlib inline
Pparams, conv_t, conv_f, text = simulate_s_params(resonator_design, "Feedline", selection=[], open_pins=[], port_list=[('P1_Q', 'in', 50), ('P2_Q', 'in', 50)], box_plus_buffer=True)

{'name': 'Sweep_DrivenModal_setup', 'reuse_selected_design': True, 'reuse_setup': True, 'freq_ghz': 6.0, 'max_delta_s': 0.005, 'max_passes': 20, 'min_passes': 2, 'min_converged': 1, 'pct_refinement': 30, 'basis_order': -1, 'vars': {'Lj': '10 nH', 'Cj': '0 fF'}, 'sweep_setup': {'name': 'Sweep', 'start_ghz': 2.0, 'stop_ghz': 8.0, 'count': 101, 'step_ghz': None, 'type': 'Fast', 'save_fields': False}}
{'name': 'Sweep_options_dm_sweep', 'start_ghz': 2.0, 'stop_ghz': 10.0, 'count': 10001, 'step_ghz': None, 'type': 'Interpolating', 'save_fields': False}


INFO 12:25PM [connect_project]: Connecting to Ansys Desktop API...
INFO 12:25PM [load_ansys_project]: 	Opened Ansys App
INFO 12:25PM [load_ansys_project]: 	Opened Ansys Desktop v2025.1.0
INFO 12:25PM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/DELL/Documents/Ansoft/
	Project:   Project240
INFO 12:25PM [connect_design]: No active design found (or error getting active design).
INFO 12:25PM [connect]: 	 Connected to project "Project240". No design detected
INFO 12:26PM [connect_design]: 	Opened active design
	Design:    Sweep_DrivenModal_hfss [Solution type: DrivenModal]
WARNING 12:26PM [connect_setup]: 	No design setup detected.
WARNING 12:26PM [connect_setup]: 	Creating driven modal default setup.
INFO 12:26PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssDMSetup'>)
INFO 12:26PM [get_setup]: 	Opened setup `Sweep_DrivenModal_setup`  (<class 'pyEPR.ansys.HfssDMSetup'>)
INFO 12:26PM [get_setup]: 	Opened setup `Sweep_DrivenModal_setup`  (<class 'pyEPR.ans

# Perfect Design

In [ ]:
# Import useful packages
import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, open_docs
from qiskit_metal.toolbox_metal import math_and_overrides
from qiskit_metal.qlibrary.core import QComponent
from collections import OrderedDict

# To create plots after geting solution data.
import matplotlib.pyplot as plt
import numpy as np

# Packages for the simple design
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import (
    LaunchpadWirebondDriven,
)
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee

# Analysis
# from qiskit_metal.renderers.renderer_gds.gds_renderer import QGDSRenderer
# from qiskit_metal.analyses.quantization import EPRanalysis
from qiskit_metal.analyses.quantization import EPRanalysis
from qiskit_metal.analyses.simulation import ScatteringImpedanceSim
from qiskit_metal.analyses.sweep_and_optimize.sweeping import Sweeping
import pyEPR as epr

In [ ]:
# Set up chip dimensions
design = designs.DesignPlanar()
design._chips["main"]["size"]["size_x"] = "9mm"
design._chips["main"]["size"]["size_y"] = "9mm"
design._chips["main"]["size"]["size_z"] = "-280um"
# Resonator and feedline gap width (W) and center conductor width (S) from reference 2
design.variables["cpw_width"] = "10 um"  # S from reference 2
design.variables["cpw_gap"] = "6 um"  # W from reference 2


design.overwrite_enabled = True

hfss = design.renderers.hfss

# Open GUI

In [ ]:
###################
# Single feedline #
###################

# Driven Lauchpad 1
x = "-1.5mm"
y = "2.0mm"
launch_options = dict(
    chip="main", pos_x=x, pos_y=y, orientation="360", lead_length="30um"
)
LP1 = LaunchpadWirebondDriven(design, "LP1", options=launch_options)

# Driven Launchpad 2
x = "1.5mm"
y = "2.0mm"
launch_options = dict(
    chip="main", pos_x=x, pos_y=y, orientation="180", lead_length="30um"
)
LP2 = LaunchpadWirebondDriven(design, "LP2", options=launch_options)

# # coupling resonator to feedline
# q_read = CoupledLineTee(
#     design,
#     "Q_Read_T",
#     options=dict(
#         pos_x="0.0mm",
#         pos_y="2mm",
#         orientation="0",
#         coupling_space="6um",
#         coupling_length="300um",
#         open_termination=False,
#     ),
# )




In [ ]:
# Using path finder to connect the two launchpads
# TL_LP1_T = RoutePathfinder(
#     design,
#     "TL_LP1_T",
#     options=dict(
#         chip="main",
#         trace_width="10um",
#         trace_gap="6um",
#         fillet="99um",
#         hfss_wire_bonds=True,
#         lead=dict(end_straight="0.1mm"),
#         pin_inputs=Dict(
#             start_pin=Dict(component="LP1", pin="tie"),
#             end_pin=Dict(component="Q_Read_T", pin="prime_start"),
#         ),
#     ),
# )

# TL_T_LP2 = RoutePathfinder(
#     design,
#     "TL_T_LP2",
#     options=dict(
#         chip="main",
#         trace_width="10um",
#         trace_gap="6um",
#         fillet="99um",
#         hfss_wire_bonds=True,
#         lead=dict(end_straight="0.1mm"),
#         pin_inputs=Dict(
#             start_pin=Dict(component="Q_Read_T", pin="prime_end"),
#             end_pin=Dict(component="LP2", pin="tie"),
#         ),
#     ),
# )
# # # Rebuild the GUI
TL_LP1_LP2 = RoutePathfinder(
    design,
    "TL_LP1_LP2",
    options=dict(
        chip="main",
        trace_width="10um",
        trace_gap="6um",
        fillet="99um",
        hfss_wire_bonds=True,
        lead=dict(end_straight="0.1mm"),
        pin_inputs=Dict(
            start_pin=Dict(component="LP1", pin="tie"),
            end_pin=Dict(component="LP2", pin="tie"),
        ),
    ),
)


In [ ]:
# ######################
# # lambda/4 resonator #
# ######################

# # First we define the two end-points
# otg = OpenToGround(
#     design,
#     "otg",
#     options=dict(chip="main", pos_x="0.0mm", pos_y="0.8mm", orientation="-90"),
# )

# # Use RouteMeander to fix the total length of the resonator
# rt_meander = RouteMeander(
#     design,
#     "meander",
#     Dict(
#         trace_width="10um",
#         trace_gap="6um",
#         total_length="3.7mm",
#         hfss_wire_bonds=True,
#         fillet="99 um",
#         lead=dict(start_straight="250um"),
#         pin_inputs=Dict(
#             start_pin=Dict(component="otg", pin="open"),
#             end_pin=Dict(component="Q_Read_T", pin="second_end"),
#         ),
#     ),
# )

# # rebuild the GUI


In [ ]:
metal.view(design)

In [ ]:
from qiskit_metal.analyses.simulation import ScatteringImpedanceSim

em1 = ScatteringImpedanceSim(design, "hfss")

In [ ]:
design_name = "Sweep_DrivenModal"
qcomp_render = []  # Means to render everything in qgeometry table.
open_terminations = []

# Here, pin LP1_in and LP2_in are converted into lumped ports,
#           each with an impedance of 50 Ohms. <br>
port_list = [("LP1", "in", 50), ("LP2", "in", 50)]
box_plus_buffer = True

In [ ]:
em1.setup.name = "Sweep_DrivenModal_setup"
em1.setup.freq_ghz = 6.0  # Try to keep this at the center of the swept frequency range for 'fast' sweeps and at the largest frequency for interpolating sweep for the best results
em1.setup.max_delta_s = 0.005  # This is necessary to get good results if interpolating sweep is not working for you
em1.setup.max_passes = 18
em1.setup.min_passes = 2
em1.setup.basis_order = -1  # Mixed order
em1.setup

In [ ]:
# we use HFSS as rendere
hfss = em1.renderer
hfss.start()

In [ ]:
# set buffer
hfss.options["x_buffer_width_mm"] = 0.1
hfss.options["y_buffer_width_mm"] = 0.1

In [ ]:
# clean the design if needed
# hfss.clean_active_design()

In [ ]:
# render the design
# em1._render(
#     selection=[],
#     solution_type="drivenmodal",
#     vars_to_initialize=em1.setup.vars,
#     open_pins=open_terminations,
#     port_list=port_list,
#     box_plus_buffer=box_plus_buffer,
# )

# render the design
em1._render(
    name = "Sweep_DrivenModal",
    selection=[],
    solution_type="drivenmodal",
    vars_to_initialize=em1.setup.vars,
    open_pins=[],
    port_list=[('LP1', 'in', 50), ('LP2', 'in', 50)],
    box_plus_buffer=box_plus_buffer,
)

In [ ]:
em1.close()

In [ ]:
# for accurate simulations, make sure the mesh is fine enough for the meander
# hfss.modeler.mesh_length("cpw_mesh", ["trace_meander"], MaxLength="0.05mm")

In [ ]:
em1.setup.sweep_setup.start_ghz = 4.0
em1.setup.sweep_setup.stop_ghz = 8.0
em1.setup.sweep_setup.count = 10001
em1.setup.sweep_setup.type = "Interpolating"
em1._analyze()  # This is necessary to keep the changes made to max_delta_s and min_passes

In [ ]:
%matplotlib inline
hfss.plot_params(["S11", "S21"])

In [ ]:
hfss.get_convergences()  # Make sure that it converges

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
s_values = hfss.get_params(['S11', 'S21'])
s_values
ax.plot(s_values[0]/1e9, np.abs(s_values[1][0]) + np.abs(s_values[1][1]) )
# ax.set_ylim(0.99,1.05)
ax.grid(True)

In [ ]:
import matplotlib.pyplot as plt
def plot_multiple_complex_arrays(*complex_arrays, labels=None):
    """
    Plots multiple arrays of complex numbers on a single 2D plane.
    
    Parameters:
    *complex_arrays : Arbitrary number of lists containing complex numbers.
    labels (list)   : Optional list of strings to label each array in the legend.
    """
    plt.figure(figsize=(9, 9))
    
    # Iterate through each array passed to the function
    for i, complex_numbers in enumerate(complex_arrays):
        real_parts = [z.real for z in complex_numbers]
        imag_parts = [z.imag for z in complex_numbers]
        
        # Determine the label for the legend
        if labels and i < len(labels):
            current_label = labels[i]
        else:
            current_label = f'Array {i + 1}'
            
        # Plot this specific array (Matplotlib automatically assigns a new color)
        plt.scatter(real_parts, imag_parts, marker='o', label=current_label, zorder=3, s=1)

        # Annotate each point with its mathematical value
        # for z in complex_numbers:
        #     point_label = f"{z.real:g}{'+' if z.imag >= 0 else ''}{z.imag:g}j"
        #     plt.annotate(point_label, (z.real, z.imag), textcoords="offset points", 
        #                  xytext=(5, 5), ha='left', fontsize=8)

    # Configure axes and grid
    plt.axhline(0, color='black', linewidth=1.5, zorder=1) # X-axis
    plt.axvline(0, color='black', linewidth=1.5, zorder=1) # Y-axis
    plt.grid(True, linestyle='--', alpha=0.6, zorder=0)

    # Set labels, title, and legend
    plt.xlabel('Real Axis')
    plt.ylabel('Imaginary Axis')
    plt.title('Complex Plane (Multiple Arrays)')
    plt.legend()

    # Ensure the aspect ratio is 1:1
    plt.axis('equal')

    # Display the plot
    plt.show()

plot_multiple_complex_arrays(s_values[1][0], s_values[1][1], labels=['S11', 'S21'])